# AdS4 Phase-Lock v18.1: Multi-Slice Discriminator

**Goal:** upgrade v17 *indistinguishable* into a stronger joint test.

This notebook compares:

1. **single-slice fits** — one model per `c` slice  
2. **joint multi-slice fit** — one shared backbone across all slices

If ambiguity survives single-slice fitting but weakens under shared multi-slice structure, then the remaining branch freedom is **global**, not just a local optimizer artifact.

## Notebook status

This is a **repo-ready v18.1 scaffold**:

- shared backbone + multi-head architecture
- EE / WL / GEO / metric outputs
- cross-slice consistency penalty
- single-slice baseline
- held-out interpolation test
- diagnostic plots

The target functions below are **synthetic placeholders** so the notebook runs immediately.
Replace them with your exact v17 observables / reconstruction targets when you are ready.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Hyperparameters

In [ ]:
# Control slices
c_values = [0.00, 0.16, 0.30]

# Grids
n_x = 160
n_r = 160
x_min, x_max = -2.0, 2.0
r_min, r_max = 0.2, 3.0

# Training
epochs_joint = 2500
epochs_single = 1800
epochs_heldout = 2200
lr = 1e-3

# Architecture
hidden = 64
depth = 3

# Loss weights
w_ee = 1.0
w_wl = 1.0
w_geo = 1.0
w_metric = 1.0
w_smooth = 1e-4
w_consistency = 0.5

# Logging
print_every = 250

## Synthetic targets

These are stand-in targets so the notebook works out of the box.

Replace these four target generators with your exact v17 definitions when you move from scaffold to experiment.

In [ ]:
def true_metric(r, c):
    # Placeholder family with c-dependent structure
    return 1.0 + r**2 - (1.0 + 0.8*c) / (r + 0.25) + 0.06 * c * np.sin(2.5 * r)

def true_ee(x, c):
    return np.log(1.0 + x**2) + 0.10 * c * np.exp(-x**2) + 0.02 * np.cos(2*x)

def true_wl(x, c):
    return np.sqrt(1.0 + x**2) + 0.08 * c / (1.0 + x**2) + 0.015 * np.sin(3*x)

def true_geo(x, c):
    return 1.0 / (1.0 + x**2) + 0.12 * c * x**2 / (1.0 + x**2) + 0.01 * np.cos(4*x)

## Data assembly

In [ ]:
x_grid = np.linspace(x_min, x_max, n_x).astype(np.float32)
r_grid = np.linspace(r_min, r_max, n_r).astype(np.float32)

targets = {}
for c in c_values:
    targets[c] = {
        "x": torch.tensor(x_grid, device=device).unsqueeze(1),
        "r": torch.tensor(r_grid, device=device).unsqueeze(1),
        "ee": torch.tensor(true_ee(x_grid, c), device=device).unsqueeze(1),
        "wl": torch.tensor(true_wl(x_grid, c), device=device).unsqueeze(1),
        "geo": torch.tensor(true_geo(x_grid, c), device=device).unsqueeze(1),
        "metric": torch.tensor(true_metric(r_grid, c), device=device).unsqueeze(1),
    }

print("Slices:", list(targets.keys()))
print("x shape:", targets[c_values[0]]["x"].shape, "| r shape:", targets[c_values[0]]["r"].shape)

## Model

### v18.1 architecture
- **shared observable backbone**
- **shared metric backbone**
- task-specific heads for:
  - EE
  - WL
  - GEO
  - metric

The shared backbone is the discriminator move:
one latent structure must explain multiple slices, rather than letting every slice drift independently.

In [ ]:
class SharedBackbone(nn.Module):
    def __init__(self, in_dim=2, hidden=64, depth=3):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

class SliceHead(nn.Module):
    def __init__(self, hidden=64, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, h):
        return self.net(h)

class MultiSliceModel(nn.Module):
    def __init__(self, hidden=64, depth=3):
        super().__init__()
        self.backbone_obs = SharedBackbone(in_dim=2, hidden=hidden, depth=depth)
        self.backbone_metric = SharedBackbone(in_dim=2, hidden=hidden, depth=depth)

        self.ee_head = SliceHead(hidden, 1)
        self.wl_head = SliceHead(hidden, 1)
        self.geo_head = SliceHead(hidden, 1)
        self.metric_head = SliceHead(hidden, 1)

    def forward_obs(self, x, c):
        c_tensor = torch.full_like(x, float(c))
        z = torch.cat([x, c_tensor], dim=1)
        h = self.backbone_obs(z)
        return self.ee_head(h), self.wl_head(h), self.geo_head(h)

    def forward_metric(self, r, c):
        c_tensor = torch.full_like(r, float(c))
        z = torch.cat([r, c_tensor], dim=1)
        h = self.backbone_metric(z)
        return self.metric_head(h)

## Loss functions

In [ ]:
mse = nn.MSELoss()

def smoothness_loss(y):
    return ((y[1:] - y[:-1])**2).mean()

def compute_slice_loss(model, batch, c):
    pred_ee, pred_wl, pred_geo = model.forward_obs(batch["x"], c)
    pred_metric = model.forward_metric(batch["r"], c)

    loss_ee = mse(pred_ee, batch["ee"])
    loss_wl = mse(pred_wl, batch["wl"])
    loss_geo = mse(pred_geo, batch["geo"])
    loss_metric = mse(pred_metric, batch["metric"])
    loss_smooth = smoothness_loss(pred_metric)

    total = (
        w_ee * loss_ee +
        w_wl * loss_wl +
        w_geo * loss_geo +
        w_metric * loss_metric +
        w_smooth * loss_smooth
    )

    parts = {
        "ee": float(loss_ee.item()),
        "wl": float(loss_wl.item()),
        "geo": float(loss_geo.item()),
        "metric": float(loss_metric.item()),
        "smooth": float(loss_smooth.item()),
        "total": float(total.item()),
    }
    return total, parts

def backbone_consistency_loss(model, x_probe, c1, c2):
    c1_tensor = torch.full_like(x_probe, float(c1))
    c2_tensor = torch.full_like(x_probe, float(c2))
    z1 = torch.cat([x_probe, c1_tensor], dim=1)
    z2 = torch.cat([x_probe, c2_tensor], dim=1)

    h1 = model.backbone_obs(z1)
    h2 = model.backbone_obs(z2)

    return ((h1 - h2)**2).mean()

## Single-slice baseline

Train one model per slice independently.

This is important because v18.1 is only meaningful if it beats or qualitatively sharpens the single-slice story.

In [ ]:
def train_single_slice(c, epochs=epochs_single, lr=lr, hidden=hidden, depth=depth):
    model = MultiSliceModel(hidden=hidden, depth=depth).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)

    history = {
        "total": [],
        "ee": [],
        "wl": [],
        "geo": [],
        "metric": [],
    }

    for epoch in range(epochs):
        opt.zero_grad()
        loss, parts = compute_slice_loss(model, targets[c], c)
        loss.backward()
        opt.step()

        history["total"].append(parts["total"])
        history["ee"].append(parts["ee"])
        history["wl"].append(parts["wl"])
        history["geo"].append(parts["geo"])
        history["metric"].append(parts["metric"])

        if epoch % print_every == 0 or epoch == epochs - 1:
            print(f"[single c={c:.2f}] epoch {epoch:4d} | total {parts['total']:.6e}")

    return model, history

single_models = {}
single_histories = {}
for c in c_values:
    print("\nTraining single-slice baseline for c =", c)
    model_c, hist_c = train_single_slice(c)
    single_models[c] = model_c
    single_histories[c] = hist_c

## Joint multi-slice fit

One shared model must fit all slices at once.

This is the actual v18.1 discriminator test.

In [ ]:
def train_joint(c_train, epochs=epochs_joint, lr=lr, hidden=hidden, depth=depth, verbose=True):
    model = MultiSliceModel(hidden=hidden, depth=depth).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)

    history = {
        "total": [],
        "consistency": [],
        "slice_total": {c: [] for c in c_train},
        "slice_metric": {c: [] for c in c_train},
    }

    x_probe = torch.linspace(-1.5, 1.5, 100, device=device).unsqueeze(1)

    for epoch in range(epochs):
        opt.zero_grad()

        total_loss = 0.0
        slice_parts = {}

        for c in c_train:
            loss_c, parts = compute_slice_loss(model, targets[c], c)
            total_loss = total_loss + loss_c
            slice_parts[c] = parts

        consistency = 0.0
        for i in range(len(c_train) - 1):
            consistency = consistency + backbone_consistency_loss(model, x_probe, c_train[i], c_train[i+1])

        total_loss = total_loss + w_consistency * consistency
        total_loss.backward()
        opt.step()

        history["total"].append(float(total_loss.item()))
        history["consistency"].append(float(consistency.item()) if torch.is_tensor(consistency) else float(consistency))
        for c in c_train:
            history["slice_total"][c].append(slice_parts[c]["total"])
            history["slice_metric"][c].append(slice_parts[c]["metric"])

        if verbose and (epoch % print_every == 0 or epoch == epochs - 1):
            print(f"[joint] epoch {epoch:4d} | total {history['total'][-1]:.6e} | consistency {history['consistency'][-1]:.6e}")

    return model, history

print("\nTraining joint multi-slice model...")
joint_model, joint_history = train_joint(c_values)

## Diagnostics helpers

In [ ]:
def metric_mse(model, c):
    with torch.no_grad():
        pred = model.forward_metric(targets[c]["r"], c)
        return float(mse(pred, targets[c]["metric"]).item())

def obs_mse(model, c):
    with torch.no_grad():
        pred_ee, pred_wl, pred_geo = model.forward_obs(targets[c]["x"], c)
        return {
            "ee": float(mse(pred_ee, targets[c]["ee"]).item()),
            "wl": float(mse(pred_wl, targets[c]["wl"]).item()),
            "geo": float(mse(pred_geo, targets[c]["geo"]).item()),
        }

def print_comparison_table():
    print("Comparison: single-slice vs joint")
    print("-" * 72)
    print(f"{'c':>6} | {'single metric':>14} | {'joint metric':>14} | {'joint obs avg':>14}")
    print("-" * 72)
    for c in c_values:
        s_metric = metric_mse(single_models[c], c)
        j_metric = metric_mse(joint_model, c)
        j_obs = obs_mse(joint_model, c)
        j_obs_avg = (j_obs["ee"] + j_obs["wl"] + j_obs["geo"]) / 3.0
        print(f"{c:6.2f} | {s_metric:14.6e} | {j_metric:14.6e} | {j_obs_avg:14.6e}")

print_comparison_table()

## Loss curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(joint_history["total"], label="joint total")
plt.plot(joint_history["consistency"], label="consistency")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("v18.1 joint training losses")
plt.legend()
plt.show()

In [ ]:
for c in c_values:
    plt.figure(figsize=(8, 4))
    plt.plot(single_histories[c]["total"], label=f"single c={c:.2f}")
    plt.yscale("log")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"Single-slice baseline loss at c={c:.2f}")
    plt.legend()
    plt.show()

## Observable fits for the joint model

In [ ]:
for c in c_values:
    x = targets[c]["x"].detach().cpu().numpy().flatten()

    with torch.no_grad():
        pred_ee, pred_wl, pred_geo = joint_model.forward_obs(targets[c]["x"], c)

    plt.figure(figsize=(9, 5))
    plt.plot(x, targets[c]["ee"].detach().cpu().numpy(), label=f"EE true c={c}")
    plt.plot(x, pred_ee.detach().cpu().numpy(), "--", label="EE pred")
    plt.plot(x, targets[c]["wl"].detach().cpu().numpy(), label="WL true")
    plt.plot(x, pred_wl.detach().cpu().numpy(), "--", label="WL pred")
    plt.plot(x, targets[c]["geo"].detach().cpu().numpy(), label="GEO true")
    plt.plot(x, pred_geo.detach().cpu().numpy(), "--", label="GEO pred")
    plt.xlabel("x")
    plt.ylabel("observable")
    plt.title(f"Joint observable fit at c={c}")
    plt.legend()
    plt.show()

## Metric reconstruction: single vs joint

In [ ]:
for c in c_values:
    r = targets[c]["r"].detach().cpu().numpy().flatten()

    with torch.no_grad():
        pred_joint = joint_model.forward_metric(targets[c]["r"], c)
        pred_single = single_models[c].forward_metric(targets[c]["r"], c)

    plt.figure(figsize=(8, 5))
    plt.plot(r, targets[c]["metric"].detach().cpu().numpy(), label=f"true metric c={c}")
    plt.plot(r, pred_single.detach().cpu().numpy(), "--", label="single pred")
    plt.plot(r, pred_joint.detach().cpu().numpy(), ":", label="joint pred")
    plt.xlabel("r")
    plt.ylabel("metric")
    plt.title(f"Metric comparison at c={c}")
    plt.legend()
    plt.show()

## Latent backbone diagnostics

This plot helps inspect whether the shared observable backbone stays coherent across slices.

In [ ]:
with torch.no_grad():
    x_probe = torch.linspace(-1.5, 1.5, 120, device=device).unsqueeze(1)
    latents = {}
    for c in c_values:
        c_tensor = torch.full_like(x_probe, float(c))
        z = torch.cat([x_probe, c_tensor], dim=1)
        latents[c] = joint_model.backbone_obs(z).detach().cpu().numpy()

for c in c_values:
    plt.figure(figsize=(9, 4))
    plt.imshow(latents[c].T, aspect="auto", origin="lower")
    plt.xlabel("probe index")
    plt.ylabel("latent channel")
    plt.title(f"Joint observable backbone latent map at c={c}")
    plt.colorbar()
    plt.show()

## Held-out interpolation test

Train on the outer slices only:
- train on `c = 0.00` and `c = 0.30`
- evaluate on the held-out middle slice `c = 0.16`

This is useful because interpolation is a stronger global-structure test than per-slice fitting alone.

In [ ]:
train_slices = [0.00, 0.30]
heldout_c = 0.16

print(f"Training held-out model on slices {train_slices}, evaluating at held-out c={heldout_c:.2f}")
heldout_model, heldout_history = train_joint(train_slices, epochs=epochs_heldout, verbose=True)

heldout_metric_err = metric_mse(heldout_model, heldout_c)
heldout_obs_errs = obs_mse(heldout_model, heldout_c)
heldout_obs_avg = (heldout_obs_errs["ee"] + heldout_obs_errs["wl"] + heldout_obs_errs["geo"]) / 3.0

print("\nHeld-out evaluation")
print("-" * 48)
print(f"held-out c          : {heldout_c:.2f}")
print(f"metric MSE          : {heldout_metric_err:.6e}")
print(f"EE / WL / GEO MSE   : {heldout_obs_errs}")
print(f"observable avg MSE  : {heldout_obs_avg:.6e}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(heldout_history["total"], label="held-out training total")
plt.plot(heldout_history["consistency"], label="held-out consistency")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Held-out interpolation training losses")
plt.legend()
plt.show()

In [ ]:
x = targets[heldout_c]["x"].detach().cpu().numpy().flatten()
r = targets[heldout_c]["r"].detach().cpu().numpy().flatten()

with torch.no_grad():
    pred_ee, pred_wl, pred_geo = heldout_model.forward_obs(targets[heldout_c]["x"], heldout_c)
    pred_metric = heldout_model.forward_metric(targets[heldout_c]["r"], heldout_c)

plt.figure(figsize=(9, 5))
plt.plot(x, targets[heldout_c]["ee"].detach().cpu().numpy(), label="EE true")
plt.plot(x, pred_ee.detach().cpu().numpy(), "--", label="EE pred")
plt.plot(x, targets[heldout_c]["wl"].detach().cpu().numpy(), label="WL true")
plt.plot(x, pred_wl.detach().cpu().numpy(), "--", label="WL pred")
plt.plot(x, targets[heldout_c]["geo"].detach().cpu().numpy(), label="GEO true")
plt.plot(x, pred_geo.detach().cpu().numpy(), "--", label="GEO pred")
plt.xlabel("x")
plt.ylabel("observable")
plt.title("Held-out observable fit at c=0.16")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(r, targets[heldout_c]["metric"].detach().cpu().numpy(), label="metric true")
plt.plot(r, pred_metric.detach().cpu().numpy(), "--", label="metric pred")
plt.xlabel("r")
plt.ylabel("metric")
plt.title("Held-out metric fit at c=0.16")
plt.legend()
plt.show()

## Interpretation guide

### If v18.1 succeeds
Look for:
- joint fit staying competitive with single-slice fits
- smoother latent structure across slices
- held-out middle slice predicted reasonably well
- reduced evidence of arbitrary branch swapping between slices

### If v18.1 fails
That is still useful.

It means:
- the current probe family remains underconstrained even under shared multi-slice structure
- the next step should be **v19 = minimal separating probe**

## Suggested v19

**`ads4_phase_lock_v19_minimal_separating_probe.ipynb`**

Search for the smallest added observable or consistency condition that breaks the remaining ambiguity.

## Final notes for repo use

When converting this from scaffold to the real experiment:

1. Replace `true_metric`, `true_ee`, `true_wl`, `true_geo` with exact v17 targets  
2. Keep the single-slice vs joint comparison  
3. Keep the held-out interpolation test  
4. Save representative figures for README / tweet / paper notes  
5. If needed, split the notebook later into:
   - v18.1 discriminator
   - v18.2 held-out interpolation
   - v19 minimal separating probe